# Gold Layer — Feature Engineering & Star Schema
---
**Goal:** Build features and Star Schema with clean column names
compatible with Power BI, Microsoft Fabric, and local Python.

**Input:** Silver_Layer/US_Accidents_Silver.csv
**Output:** Gold_Layer/ — Fact Table + 7 Dim tables + Modeling file + Road_Feature_Severity.csv

> Column names use underscores only — no parentheses or special characters.

## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('Gold_Layer', exist_ok=True)

# Column rename map — clean names compatible with Fabric, Power BI, Python
RENAME_MAP = {
    'Temperature(F)'   : 'Temperature_F',
    'Humidity(%)'      : 'Humidity_pct',
    'Pressure(in)'     : 'Pressure_in',
    'Visibility(mi)'   : 'Visibility_mi',
    'Wind_Speed(mph)'  : 'Wind_Speed_mph',
    'Precipitation(in)': 'Precipitation_in',
    'Wind_Chill(F)'    : 'Wind_Chill_F',
    'Distance(mi)'     : 'Distance_mi',
}
print('Setup done | Clean column names ready')

Setup done | Clean column names ready


## 2. Load Silver Data

In [2]:
df = pd.read_csv('Silver_Layer/US_Accidents_Silver.csv', low_memory=False)
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce', format='mixed')
df['End_Time']   = pd.to_datetime(df['End_Time'],   errors='coerce', format='mixed')

# Apply clean column names immediately
df = df.rename(columns=RENAME_MAP)

print(f'Loaded: {df.shape[0]:,} rows | {df.shape[1]} columns')
print('Renamed columns:')
for old, new in RENAME_MAP.items():
    if new in df.columns:
        print(f'  {old:<20} -> {new}')

Loaded: 500,000 rows | 46 columns
Renamed columns:
  Temperature(F)       -> Temperature_F
  Humidity(%)          -> Humidity_pct
  Pressure(in)         -> Pressure_in
  Visibility(mi)       -> Visibility_mi
  Wind_Speed(mph)      -> Wind_Speed_mph
  Precipitation(in)    -> Precipitation_in
  Wind_Chill(F)        -> Wind_Chill_F
  Distance(mi)         -> Distance_mi


## 3. Datetime Features
Duration / Hour / Day / Month / Year / Weekend / Rush Hour / Time of Day

In [3]:
df['Duration_Minutes'] = (
    (df['End_Time'] - df['Start_Time'])
    .dt.total_seconds().div(60).clip(lower=0))
df['Hour']         = df['Start_Time'].dt.hour
df['Day_of_Week']  = df['Start_Time'].dt.dayofweek
df['Month']        = df['Start_Time'].dt.month
df['Year']         = df['Start_Time'].dt.year
df['Day_of_Month'] = df['Start_Time'].dt.day
df['Week_of_Year'] = df['Start_Time'].dt.isocalendar().week.astype(int)
df['Is_Weekend']   = (df['Day_of_Week'] >= 5).astype(int)
df['Date']         = df['Start_Time'].dt.date
df['Is_Rush_Hour'] = (
    ((df['Hour'] >= 7) & (df['Hour'] <= 9)) |
    ((df['Hour'] >= 16) & (df['Hour'] <= 18))).astype(int)
df['Time_of_Day'] = pd.cut(
    df['Hour'], bins=[-1,5,11,16,20,23],
    labels=['Night','Morning','Afternoon','Evening','Night2'])
df['Time_of_Day'] = df['Time_of_Day'].astype(str).replace('Night2','Night')
print(f'Datetime features done | Rush hour: {df["Is_Rush_Hour"].sum():,}')

Datetime features done | Rush hour: 201,589


## 4. Weather Features
Extreme / Low Visibility / Rain / Snow / Fog / Visibility Category / Temp Category

In [4]:
df['Wind_Chill_Effect']   = df['Temperature_F'] - df['Wind_Chill_F']
df['Is_Extreme_Weather']  = (
    (df['Visibility_mi'] <= 1.0) |
    (df['Precipitation_in'] > 0.1) |
    (df['Weather_Condition'].str.contains('Snow|Heavy', na=False))).astype(int)
df['Is_Low_Visibility']   = (df['Visibility_mi'] <= 2).astype(int)
df['Is_Raining']          = df['Weather_Condition'].str.contains('Rain|Drizzle|Thunder', na=False).astype(int)
df['Is_Snowing']          = df['Weather_Condition'].str.contains('Snow|Sleet', na=False).astype(int)
df['Is_Foggy']            = df['Weather_Condition'].str.contains('Fog|Haze|Mist', na=False).astype(int)
df['Visibility_Category'] = pd.cut(
    df['Visibility_mi'], bins=[0,2,7,float('inf')],
    labels=['Low','Moderate','High'])
df['Temperature_Category'] = pd.cut(
    df['Temperature_F'],
    bins=[-float('inf'),32,50,70,85,float('inf')],
    labels=['Freezing','Cold','Cool','Warm','Hot'])
print(f'Weather features done | Extreme: {df["Is_Extreme_Weather"].sum():,}')

Weather features done | Extreme: 26,249


## 5. Road Features
Booleans → Int / Road Features Count / Traffic Signal / Crossing / Junction

In [5]:
bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].astype(int)
road_features = ['Amenity','Bump','Crossing','Give_Way','Junction','No_Exit',
                 'Railway','Roundabout','Station','Stop','Traffic_Calming','Traffic_Signal']
df['Road_Features_Count'] = df[road_features].sum(axis=1)
df['Has_Traffic_Signal']  = df['Traffic_Signal'].astype(int)
df['Has_Crossing']        = df['Crossing'].astype(int)
df['Has_Junction']        = df['Junction'].astype(int)
print(f'Road features done | Avg: {df["Road_Features_Count"].mean():.2f}')

Road features done | Avg: 0.42


## 6. Risk & Distance Features
Location Risk Score / Distance Category / Severity Binary

In [6]:
df['Location_Risk_Score'] = (
    df['Severity'] * (1 / (df['Visibility_mi'] + 0.1)) *
    (df['Precipitation_in'] + 1)).round(4)
df['Distance_Category'] = pd.cut(
    df['Distance_mi'],
    bins=[-0.001, 0.1, 0.5, 1, 5, float('inf')],
    labels=['Very_Short','Short','Medium','Long','Very_Long'])
df['Severity_Binary'] = (df['Severity'] >= 3).astype(int)
print(f'Risk & Distance done | Severity_Binary: {df["Severity_Binary"].value_counts().to_dict()}')

# Severity label — clear text description for Power BI and presentations
severity_label_map = {
    1: 'Low Impact',
    2: 'Moderate Impact',
    3: 'High Impact',
    4: 'Critical Impact',
}
df['Severity_Label'] = df['Severity'].map(severity_label_map)
print(f'Severity labels added:')
print(df['Severity_Label'].value_counts().to_string())

Risk & Distance done | Severity_Binary: {0: 402416, 1: 97584}
Severity labels added:
Severity_Label
Moderate Impact    398142
High Impact         84520
Critical Impact     13064
Low Impact           4274


## 7. Encoding
Convert categorical columns to numeric for ML

In [7]:
df['Time_of_Day_Encoded']          = df['Time_of_Day'].map({'Night':0,'Morning':1,'Afternoon':2,'Evening':3})
df['Temperature_Category_Encoded'] = df['Temperature_Category'].map({'Freezing':0,'Cold':1,'Cool':2,'Warm':3,'Hot':4})
df['Sunrise_Sunset_Encoded']       = df['Sunrise_Sunset'].map({'Day':1,'Night':0}).fillna(0).astype(int)
df['Distance_Category_Encoded']    = df['Distance_Category'].map(
    {'Very_Short':0,'Short':1,'Medium':2,'Long':3,'Very_Long':4}).fillna(0).astype(int)
for col in ['Timezone','Wind_Direction','Weather_Condition']:
    df[col+'_Encoded'] = df[col].astype('category').cat.codes
# Fix Wind_Speed outliers
p99 = df['Wind_Speed_mph'].quantile(0.99)
df['Wind_Speed_mph'] = df['Wind_Speed_mph'].clip(upper=p99)
print(f'Encoding done | Wind_Speed_mph max: {df["Wind_Speed_mph"].max():.1f} mph')

Encoding done | Wind_Speed_mph max: 23.0 mph


## 8. Validation

In [8]:
assert df['Severity'].between(1,4).all(),              'Severity out of range!'
assert df['Duration_Minutes'].ge(0).all(),              'Negative Duration!'
assert df['Distance_Category_Encoded'].isna().sum()==0, 'Distance nulls!'
assert df['Wind_Speed_mph'].max() < 200,               'Wind_Speed outlier!'
print('All validation checks passed!')
print(f'Shape: {df.shape[0]:,} rows | {df.shape[1]} columns')

All validation checks passed!
Shape: 500,000 rows | 80 columns


## 9. Build Dimension Tables

In [9]:
# Dim_Date
dim_date = df[['Date']].drop_duplicates().copy()
dim_date['Year']        = pd.to_datetime(dim_date['Date']).dt.year
dim_date['Month']       = pd.to_datetime(dim_date['Date']).dt.month
dim_date['Day']         = pd.to_datetime(dim_date['Date']).dt.day
dim_date['Day_of_Week'] = pd.to_datetime(dim_date['Date']).dt.day_name()
dim_date['Is_Weekend']  = dim_date['Day_of_Week'].isin(['Saturday','Sunday']).astype(int)
dim_date.insert(0,'Date_ID', range(1, len(dim_date)+1))

loc_cols     = ['Street','City','County','State','Zipcode','Timezone']
dim_location = df[loc_cols].drop_duplicates().reset_index(drop=True)
dim_location.insert(0,'Location_ID', range(1, len(dim_location)+1))

road_cols = ['Amenity','Bump','Crossing','Give_Way','Junction','No_Exit','Railway','Roundabout','Station','Stop','Traffic_Calming','Traffic_Signal','Turning_Loop']
dim_road  = df[road_cols].drop_duplicates().reset_index(drop=True)
dim_road.insert(0,'Road_Feature_ID', range(1, len(dim_road)+1))

weather_cols = ['Weather_Condition','Is_Extreme_Weather']
dim_weather  = df[weather_cols].drop_duplicates().reset_index(drop=True)
dim_weather.insert(0,'Weather_Condition_ID', range(1, len(dim_weather)+1))

twi_cols     = ['Sunrise_Sunset','Civil_Twilight','Nautical_Twilight','Astronomical_Twilight']
dim_twilight = df[twi_cols].drop_duplicates().reset_index(drop=True)
dim_twilight.insert(0,'Twilight_ID', range(1, len(dim_twilight)+1))

dim_severity = df[['Severity','Severity_Label']].drop_duplicates()\
               .sort_values('Severity').reset_index(drop=True)
dim_severity.insert(0,'Severity_ID', range(1, len(dim_severity)+1))

dim_source = df[['Source']].drop_duplicates().reset_index(drop=True)
dim_source.insert(0,'Source_ID', range(1, len(dim_source)+1))

for name, dim in [('Date',dim_date),('Location',dim_location),('Road',dim_road),
                   ('Weather',dim_weather),('Twilight',dim_twilight),
                   ('Severity',dim_severity),('Source',dim_source)]:
    print(f'  Dim_{name}: {len(dim):,} rows')

  Dim_Date: 2,537 rows
  Dim_Location: 213,052 rows
  Dim_Road: 222 rows
  Dim_Weather: 162 rows
  Dim_Twilight: 10 rows
  Dim_Severity: 4 rows
  Dim_Source: 3 rows


## 10. Build Fact Table

In [10]:
gold_merged = (
    df
    .merge(dim_location, on=loc_cols, how='left')
    .merge(dim_weather,  on=weather_cols, how='left')
    .merge(dim_road,     on=road_cols, how='left')
    .merge(dim_twilight, on=twi_cols, how='left')
    .merge(
        dim_severity[['Severity', 'Severity_ID']],
        on='Severity',
        how='left'
    )
    .merge(dim_source, on='Source', how='left')
    .merge(dim_date[['Date_ID', 'Date']], on='Date', how='left')
)

fact_accidents = gold_merged[[
    'ID',
    'Severity_ID',
    'Date_ID',
    'Location_ID',
    'Weather_Condition_ID',
    'Road_Feature_ID',
    'Source_ID',
    'Twilight_ID',

    # Measures
    'Distance_mi',
    'Duration_Minutes',
    'Temperature_F',
    'Humidity_pct',
    'Pressure_in',
    'Visibility_mi',
    'Wind_Speed_mph',
    'Precipitation_in',
    'Wind_Chill_Effect',

    # Engineered features
    'Hour',
    'Day_of_Week',
    'Month',
    'Year',
    'Week_of_Year',
    'Is_Weekend',
    'Is_Rush_Hour',
    'Is_Extreme_Weather',
    'Is_Low_Visibility',
    'Is_Raining',
    'Is_Snowing',
    'Is_Foggy',
    'Road_Features_Count',
    'Has_Traffic_Signal',
    'Has_Crossing',
    'Has_Junction',
    'Location_Risk_Score',

    # Encoded
    'Time_of_Day_Encoded',
    'Temperature_Category_Encoded',
    'Sunrise_Sunset_Encoded',
    'Distance_Category_Encoded',
    'Timezone_Encoded',
    'Wind_Direction_Encoded',
    'Weather_Condition_Encoded',

    # Targets
    'Severity',
    'Severity_Label',
    'Severity_Binary'
]].copy()

print(f'Fact table: {fact_accidents.shape[0]:,} rows | {fact_accidents.shape[1]} columns')
print('All column names are clean — compatible with Power BI, Fabric, and Python')

Fact table: 500,000 rows | 44 columns
All column names are clean — compatible with Power BI, Fabric, and Python


## 11. Referential Integrity Check

In [11]:
fk_cols = ['Location_ID','Weather_Condition_ID','Road_Feature_ID',
           'Twilight_ID','Severity_ID','Source_ID','Date_ID']
all_ok = True
for fk in fk_cols:
    n = fact_accidents[fk].isna().sum()
    print(f'  {fk:<30} {"OK" if n==0 else f"FAIL: {n:,}"}')
    if n > 0: all_ok = False
print('All checks passed!' if all_ok else 'Fix orphan records!')

  Location_ID                    OK
  Weather_Condition_ID           OK
  Road_Feature_ID                OK
  Twilight_ID                    OK
  Severity_ID                    OK
  Source_ID                      OK
  Date_ID                        OK
All checks passed!


## 12. Save Gold Layer

In [12]:
# Single output — works with Power BI, Fabric, and local Python
exports = {
    'Gold_Layer/Fact_Accidents.csv':        fact_accidents,
    'Gold_Layer/Dim_Date.csv':              dim_date,
    'Gold_Layer/Dim_Location.csv':          dim_location,
    'Gold_Layer/Dim_Weather_Condition.csv': dim_weather,
    'Gold_Layer/Dim_Road_Features.csv':     dim_road,
    'Gold_Layer/Dim_Twilight.csv':          dim_twilight,
    'Gold_Layer/Dim_Source.csv':            dim_source,
    'Gold_Layer/Dim_Severity.csv':          dim_severity,
}
for path, d in exports.items():
    d.to_csv(path, index=False)
    print(f'  Saved: {path} ({len(d):,} rows)')

# ML modeling file
fact_accidents.drop(columns=['ID'], errors='ignore')\
              .to_csv('Gold_Layer/US_Accidents_Gold_Modeling.csv', index=False)
print('  Saved: Gold_Layer/US_Accidents_Gold_Modeling.csv')

print(f'''
{'='*55}
  GOLD LAYER COMPLETE
{'='*55}
  Rows    : {fact_accidents.shape[0]:,}
  Columns : {fact_accidents.shape[1]}

  Files saved to Gold_Layer/:
    Fact_Accidents.csv               <- Fact table
    Dim_Date.csv                     <- 2,537 rows
    Dim_Location.csv                 <- 213,052 rows
    Dim_Weather_Condition.csv        <- 162 rows
    Dim_Road_Features.csv            <- 222 rows
    Dim_Twilight.csv                 <- 10 rows
    Dim_Severity.csv                 <- 4 rows
    Dim_Source.csv                   <- 3 rows
    US_Accidents_Gold_Modeling.csv   <- ML file

  Column names (clean - no special chars):
    Temperature_F  | Humidity_pct   | Pressure_in
    Visibility_mi  | Wind_Speed_mph | Precipitation_in
    Distance_mi    | Wind_Chill_F

  Compatible with:
    Power BI Desktop  -> import from Gold_Layer/
    Microsoft Fabric  -> upload from Gold_Layer/
    Python / ML       -> read from Gold_Layer/
''')

  Saved: Gold_Layer/Fact_Accidents.csv (500,000 rows)
  Saved: Gold_Layer/Dim_Date.csv (2,537 rows)
  Saved: Gold_Layer/Dim_Location.csv (213,052 rows)
  Saved: Gold_Layer/Dim_Weather_Condition.csv (162 rows)
  Saved: Gold_Layer/Dim_Road_Features.csv (222 rows)
  Saved: Gold_Layer/Dim_Twilight.csv (10 rows)
  Saved: Gold_Layer/Dim_Source.csv (3 rows)
  Saved: Gold_Layer/Dim_Severity.csv (4 rows)
  Saved: Gold_Layer/US_Accidents_Gold_Modeling.csv

  GOLD LAYER COMPLETE
  Rows    : 500,000
  Columns : 44

  Files saved to Gold_Layer/:
    Fact_Accidents.csv               <- Fact table
    Dim_Date.csv                     <- 2,537 rows
    Dim_Location.csv                 <- 213,052 rows
    Dim_Weather_Condition.csv        <- 162 rows
    Dim_Road_Features.csv            <- 222 rows
    Dim_Twilight.csv                 <- 10 rows
    Dim_Severity.csv                 <- 4 rows
    Dim_Source.csv                   <- 3 rows
    US_Accidents_Gold_Modeling.csv   <- ML file

  Column names (c

## 13. Road Feature Severity Summary
Generate `Road_Feature_Severity.csv` — a pre-aggregated summary of all 13 road features
broken down by severity label. Used directly in Power BI Dashboard 5 (Road Features).

**Input:** `df` (already loaded above) — no re-read needed
**Output:** `Gold_Layer/Road_Feature_Severity.csv`

In [ ]:
# Road Feature × Severity summary — uses df already loaded in Section 2
# No re-read needed — df is available from earlier steps

ROAD_FEATURES_LIST = [
    'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction',
    'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop',
    'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop'
]

# Reuse severity_label_map defined in Section 6
rf_rows = []
for feature in ROAD_FEATURES_LIST:
    if feature not in df.columns:
        continue
    temp = df[df[feature] == 1].copy()
    grouped = (
        temp.groupby('Severity')
        .size()
        .reset_index(name='Accident_Count')
    )
    for _, r in grouped.iterrows():
        rf_rows.append({
            'Road_Feature'   : feature,
            'Severity_Label' : severity_label_map.get(r['Severity'], str(r['Severity'])),
            'Accident_Count' : int(r['Accident_Count'])
        })

road_feature_summary = pd.DataFrame(rf_rows)

out_path = 'Gold_Layer/Road_Feature_Severity.csv'
road_feature_summary.to_csv(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Rows : {len(road_feature_summary)}')
print(f'Features covered: {road_feature_summary["Road_Feature"].nunique()}')
print()
print(road_feature_summary.to_string(index=False))